In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

1. 데이터 로드

In [ ]:
customer = pd.read_csv("/train_customer_info.csv")
transaction = pd.read_csv("/train_transaction_history.csv")
finance = pd.read_csv("/train_finance_profile.csv")
target = pd.read_csv("/train_targets.csv")

* shape 확인


In [ ]:
datasets = {
    'customer'    : customer,
    'transaction' : transaction,
    'finance'     : finance,
    'target'      : target
}

for name, df in datasets.items():
    print(f"\n{'='*45}")
    print(f"  [{name}]  {df.shape[0]:,}행 × {df.shape[1]}열")
    print(f"{'='*45}")
    print(df.dtypes)


  [customer]  60,000행 × 8열
customer_id        object
join_date          object
age                 int64
gender             object
region_code        object
is_married          int64
prefer_category    object
income_group       object
dtype: object

  [transaction]  1,048,575행 × 7열
trans_id          object
customer_id       object
trans_date        object
trans_amount       int64
biz_type          object
item_category     object
is_installment     int64
dtype: object

  [finance]  60,000행 × 9열
customer_id               object
credit_score               int64
num_active_cards           int64
total_deposit_balance      int64
total_loan_balance         int64
card_cash_service_amt      int64
card_loan_amt              int64
fin_overdue_days           int64
fin_asset_trend_score    float64
dtype: object

  [target]  60,000행 × 3열
customer_id      object
target_churn      int64
target_ltv      float64
dtype: object


- 결측치 확인

In [ ]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n[{name}]")
    if missing.empty:
        print("  결측치 없음")
    else:
        pct = (missing / len(df) * 100).round(2)
        print(pd.DataFrame({'결측 수': missing, '비율(%)': pct}))


[customer]
  결측치 없음

[transaction]
  결측치 없음

[finance]
  결측치 없음

[target]
  결측치 없음


- 샘플 확인

In [ ]:
for name, df in datasets.items():
    print(f"\n[{name}]")
    display(df.head(3))


[customer]


,customer_id,join_date,age,gender,region_code,is_married,prefer_category,income_group
0,C000001,2020-04-12,36,F,R03,1,Grocery,G4
1,C000002,2021-03-11,32,F,R02,0,Home,G3
2,C000003,2022-05-10,41,M,R03,1,Fashion,G3



[transaction]


,trans_id,customer_id,trans_date,trans_amount,biz_type,item_category,is_installment
0,T00000001,C068329,2023-07-31,14855,Online,Home,0
1,T00000002,C044011,2023-10-09,40476,Offline,Fashion,0
2,T00000003,C014765,2023-11-24,186238,Online,Home,0



[finance]


,customer_id,credit_score,num_active_cards,total_deposit_balance,total_loan_balance,card_cash_service_amt,card_loan_amt,fin_overdue_days,fin_asset_trend_score
0,C000001,713,6,2517740,58608891,100572,65339,0,0.551107
1,C000002,869,5,679696,33403843,210018,0,0,-0.342776
2,C000003,588,2,17319511,0,4092,0,1,-0.949003



[target]


,customer_id,target_churn,target_ltv
0,C000001,0,556691.0
1,C000002,0,1460203.0
2,C000003,0,605476.0


파악된 내용 요약:

- 결측치 없음 → 별도 처리 불필요

- transaction이 1,048,575행 → 고객 1인당 평균 약 17건

- join_date, trans_date → 날짜 변환 필요

- gender, region_code, prefer_category, income_group, biz_type, item_category → 인코딩 필요

- 타겟 분포 확인

In [ ]:
# ── Churn 분포 ──
churn_vc  = target['target_churn'].value_counts().sort_index()
churn_pct = (churn_vc / len(target) * 100).round(2)

print("[ Churn 분포 ]")
print(pd.DataFrame({'count': churn_vc, '%': churn_pct}))
print(f"\n  → 이탈률: {target['target_churn'].mean():.2%}")

[ Churn 분포 ]
              count      %
target_churn              
0             54071  90.12
1              5929   9.88

  → 이탈률: 9.88%


In [ ]:
# ── LTV 기초통계 & 왜도 ──
ltv = target['target_ltv']

print("\n[ LTV 기초통계 ]")
print(ltv.describe().round(0))

skewness = ltv.skew()
print(f"\n  왜도(skewness): {skewness:.2f}")

if skewness > 1:
    print("  → 오른쪽 치우침 심함 log 변환 필수")
elif skewness > 0.5:
    print("  → 약간 치우침, log 변환 권장")
else:
    print("  → 정규분포에 가까움 ")

# ── log 변환 후 왜도 비교 ──
ltv_log = np.log1p(ltv)
print(f"  log 변환 후 왜도: {ltv_log.skew():.2f}")


[ LTV 기초통계 ]
count       60000.0
mean      1239290.0
std       1360034.0
min             3.0
25%        257648.0
50%        807154.0
75%       1751954.0
max      17176629.0
Name: target_ltv, dtype: float64

  왜도(skewness): 2.12
  → 오른쪽 치우침 심함 log 변환 필수
  log 변환 후 왜도: -1.14


- transaction 기본 분석

In [ ]:
# 날짜 변환
transaction['trans_date'] = pd.to_datetime(transaction['trans_date'])

# ── 거래 기간 ──
print("[ 거래 기간 ]")
print(f"  시작: {transaction['trans_date'].min().date()}")
print(f"  종료: {transaction['trans_date'].max().date()}")

# ── 월별 거래 건수 ──
print("\n[ 월별 거래 건수 ]")
transaction['month'] = transaction['trans_date'].dt.to_period('M')
print(transaction.groupby('month').size().to_frame('거래건수'))

[ 거래 기간 ]
  시작: 2023-07-01
  종료: 2023-12-31

[ 월별 거래 건수 ]
           거래건수
month          
2023-07  176934
2023-08  177007
2023-09  170360
2023-10  177448
2023-11  170588
2023-12  176238


In [ ]:
# ── 고객 1인당 거래 건수 ──
print("[ 고객 1인당 거래 건수 ]")
tx_per_customer = transaction.groupby('customer_id').size()
print(tx_per_customer.describe().round(1))

# ── biz_type / item_category 분포 ──
print("\n[ biz_type 분포 ]")
print(transaction['biz_type'].value_counts())

print("\n[ item_category 분포 ]")
print(transaction['item_category'].value_counts())

[ 고객 1인당 거래 건수 ]
count    60000.0
mean        17.5
std          4.2
min          4.0
25%         15.0
50%         17.0
75%         20.0
max         36.0
dtype: float64

[ biz_type 분포 ]
biz_type
Online     629255
Offline    419320
Name: count, dtype: int64

[ item_category 분포 ]
item_category
Electronics    210044
Grocery        209978
Beauty         209925
Fashion        209797
Home           208831
Name: count, dtype: int64


In [ ]:
# 연체일수 분포 (churn 신호 핵심 변수)
print("[ fin_overdue_days ]")
print(finance['fin_overdue_days'].describe().round(1))
print(f"  연체 있는 고객 수: {(finance['fin_overdue_days'] > 0).sum():,}명")

# 신용점수 범위 확인
print("\n[ credit_score ]")
print(finance['credit_score'].describe().round(1))

[ fin_overdue_days ]
count    60000.0
mean         0.4
std          0.6
min          0.0
25%          0.0
50%          0.0
75%          1.0
max          5.0
Name: fin_overdue_days, dtype: float64
  연체 있는 고객 수: 19,875명

[ credit_score ]
count    60000.0
mean       739.5
std        109.2
min        308.0
25%        665.0
50%        740.0
75%        814.0
max       1000.0
Name: credit_score, dtype: float64
